# SBA 7(a) Loan Risk — Final FY2019 Holdout Evaluation

This notebook evaluates the frozen Logistic Regression model
on the final FY2019 approval cohort.

## Selected model

- Logistic Regression
- 13 provisional predictors
- L2 regularization, C = 1.0
- Uncalibrated probabilities
- Training period: FY2010–2015
- Hyperparameter tuning: FY2016
- Calibration investigation: FY2017
- Independent validation and model selection: FY2018

## Final holdout

- Test cohort: FY2019
- Expected records: 45,670
- Expected charge-offs: 1,786
- Review capacity: Top 10%

## Evaluation metrics

- Average Precision and AP lift
- Recall@10%
- Precision@10%
- ROC-AUC
- Brier score and Brier skill
- Probability calibration
- Historical charge-off capture

## Methodological safeguards

No model retraining, hyperparameter tuning, feature changes,
or calibration adjustments are permitted based on FY2019 results.

The dataset represents a retrospective, status-selected
population rather than the complete historical application pool.

The selected predictors remain subject to documented
approval-time provenance limitations.

**Reproducibility guidance.** Run from the `notebooks/` working directory with Python 3.12.8 and scikit-learn 1.5.1. This notebook requires `data/exports/sba_model_export.csv`, `models/model_selection_record.json`, and the model artifact specified in that record. The saved outputs document the historical FY2019 holdout evaluation; they have **not** been regenerated during this documentation audit. One saved output still contains a previous scikit-learn 1.6.1 unpickling warning from an older environment; the notebook now checks the installed version before loading the artifact. The 2019 average predicted rate (~6.11%) exceeds the observed event rate (~3.91%), so scores should not be treated as well-calibrated default probabilities.

In [3]:
import json
import joblib
import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss
)

# Check the environment before loading serialized scikit-learn artifacts.
import sklearn
import sys
if sklearn.__version__ != "1.5.1":
    raise RuntimeError(
        f"Expected scikit-learn 1.5.1 for the saved model, found {sklearn.__version__}. "
        "Use the documented Python 3.12.8 environment."
    )

RANDOM_STATE = 42

DATA_PATH = Path("../data/exports/sba_model_export.csv")
MODEL_DIR = Path("../models")

# Confirm required local files exist before loading them.
for required_file in [DATA_PATH, MODEL_DIR / "model_selection_record.json"]:
    if not required_file.is_file():
        raise FileNotFoundError(
            f"Required file not found: {required_file}. Run from the notebooks/ folder "
            "and follow README setup instructions."
        )

# Read the previously frozen model-selection record
with open(MODEL_DIR / "model_selection_record.json", "r") as file:
    selection_record = json.load(file)

print("Selected model:", selection_record["selected_model"])
print("Selected artifact:", selection_record["selected_artifact"])
print("Test evaluated:", selection_record["test_evaluated"])

# Ensure the recorded model-selection decision is unchanged
assert selection_record["selected_model"] == "Logistic Regression"
assert selection_record["predictor_count"] == 13
assert selection_record["probability_calibration"] == "Uncalibrated"
assert selection_record["test_evaluated"] is False

print("\nFrozen selection record validated!")

Selected model: Logistic Regression
Selected artifact: logistic_regression_13_features.joblib
Test evaluated: False

Frozen selection record validated!


In [4]:
# Load the exact model specified in our selection record
model_path = MODEL_DIR / selection_record["selected_artifact"]

if not model_path.is_file():
    raise FileNotFoundError(f"Selected model artifact missing: {model_path}")

final_model = joblib.load(model_path)

# Verify the selected Logistic Regression configuration
classifier = final_model.named_steps["classifier"]

print("Model loaded:", model_path.name)
print("Model type:", type(classifier).__name__)
print("Regularization C:", classifier.C)

# Verify model configuration
assert type(classifier).__name__ == "LogisticRegression"
assert classifier.C == 1.0
assert classifier.n_features_in_ > 0

print("\nFrozen model validation passed!")

Model loaded: logistic_regression_13_features.joblib
Model type: LogisticRegression
Regularization C: 1.0

Frozen model validation passed!


/opt/anaconda3/envs/anaconda-finance/lib/python3.11/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.5.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/anaconda3/envs/anaconda-finance/lib/python3.11/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.5.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/anaconda3/envs/anaconda-finance/lib/python3.11/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from versi

In [5]:
# Load the original modeling export
df = pd.read_csv(
    DATA_PATH,
    parse_dates=["approval_date"],
    dtype={
        "sba_lender_id": "string",
        "lender_key": "string"
    }
)

# Isolate the final FY2019 holdout cohort
test_df = df.loc[
    df["modeling_period"] == "Test"
].copy()

print("Full dataset shape:", df.shape)
print("FY2019 test dataset shape:", test_df.shape)

print(
    "FY2019 charge-offs:",
    test_df["chargeoff_within_5yr"].sum()
)

print(
    "FY2019 charge-off incidence:",
    f"{test_df['chargeoff_within_5yr'].mean():.4%}"
)

# Validate test population
assert df.shape == (478953, 25)
assert len(test_df) == 45670
assert test_df["chargeoff_within_5yr"].sum() == 1786
assert test_df["approval_fy"].eq(2019).all()

print("\nFY2019 holdout cohort validation passed!")

Full dataset shape: (478953, 25)
FY2019 test dataset shape: (45670, 25)
FY2019 charge-offs: 1786
FY2019 charge-off incidence: 3.9107%

FY2019 holdout cohort validation passed!


In [6]:
numeric_features = [
    "gross_approval",
    "guarantee_pct",
    "initial_interest_rate_model",
    "jobs_supported"
]

categorical_features = [
    "business_age_group",
    "naics_sector",
    "project_state",
    "processing_method",
    "collateral_indicator",
    "revolver_status",
    "interest_rate_type",
    "business_type"
]

flag_features = [
    "invalid_interest_rate_flag"
]

feature_columns = (
    numeric_features
    + categorical_features
    + flag_features
)

target_column = "chargeoff_within_5yr"

# Verify the frozen feature selection
assert len(feature_columns) == 13
assert "term_months_model" not in feature_columns
assert "invalid_term_flag" not in feature_columns

print("Final predictor count:", len(feature_columns))
print("Feature selection validated!")

Final predictor count: 13
Feature selection validated!


In [7]:
# Select the frozen feature set
X_test = test_df[feature_columns].copy()

# Extract actual outcomes for evaluation
y_test = test_df[target_column].copy()

print("Test features:", X_test.shape)
print("Test outcomes:", y_test.shape)

# Verify the same feature names and order used during fitting
assert list(X_test.columns) == list(
    final_model.feature_names_in_
)

assert X_test.shape == (45670, 13)
assert y_test.shape == (45670,)

print("\nFinal test feature validation passed!")

Test features: (45670, 13)
Test outcomes: (45670,)

Final test feature validation passed!


In [8]:
# Generate predictions using the frozen selected model
test_probabilities = final_model.predict_proba(
    X_test
)[:, 1]

# Validate predictions
assert len(test_probabilities) == len(y_test)
assert np.isfinite(test_probabilities).all()
assert (
    (test_probabilities >= 0)
    & (test_probabilities <= 1)
).all()

print("Number of predictions:", len(test_probabilities))

print(
    "Minimum predicted risk:",
    f"{test_probabilities.min():.4%}"
)

print(
    "Maximum predicted risk:",
    f"{test_probabilities.max():.4%}"
)

print(
    "Average predicted risk:",
    f"{test_probabilities.mean():.4%}"
)

print("\nFinal holdout probability validation passed!")

Number of predictions: 45670
Minimum predicted risk: 0.0218%
Maximum predicted risk: 28.3698%
Average predicted risk: 6.1107%

Final holdout probability validation passed!


In [9]:
# Calculate observed FY2019 charge-off incidence
test_prevalence = y_test.mean()

# Calculate ranking performance
test_ap = average_precision_score(
    y_test,
    test_probabilities
)

test_roc_auc = roc_auc_score(
    y_test,
    test_probabilities
)

# Compare Average Precision with event prevalence
test_ap_lift = test_ap / test_prevalence

# Select the highest-risk 10% of loans
review_count = int(np.ceil(len(y_test) * 0.10))

top_indices = np.argsort(
    -test_probabilities,
    kind="stable"
)[:review_count]

# Count charge-offs captured in the review group
captured_chargeoffs = int(
    y_test.to_numpy()[top_indices].sum()
)

# Calculate operational performance
test_recall_10 = captured_chargeoffs / y_test.sum()

test_precision_10 = captured_chargeoffs / review_count

test_precision_lift = test_precision_10 / test_prevalence

print("FY2019 Final Holdout — Ranking Performance")
print(f"Average Precision: {test_ap:.4f}")
print(f"AP lift: {test_ap_lift:.2f}x")
print(f"ROC-AUC: {test_roc_auc:.4f}")
print(f"Recall@10%: {test_recall_10:.2%}")
print(f"Precision@10%: {test_precision_10:.2%}")
print(f"Precision lift: {test_precision_lift:.2f}x")
print("Charge-offs captured:", captured_chargeoffs)
print("Loans selected for review:", review_count)

FY2019 Final Holdout — Ranking Performance
Average Precision: 0.0774
AP lift: 1.98x
ROC-AUC: 0.7020
Recall@10%: 22.28%
Precision@10%: 8.71%
Precision lift: 2.23x
Charge-offs captured: 398
Loans selected for review: 4567


In [10]:
# Calculate original training-period prevalence
train_prevalence = df.loc[
    df["modeling_period"] == "Train",
    target_column
].mean()

# Build the reference probability predictions
reference_probabilities = np.full(
    len(y_test),
    train_prevalence
)

# Calculate Brier scores
reference_brier = brier_score_loss(
    y_test,
    reference_probabilities
)

test_brier = brier_score_loss(
    y_test,
    test_probabilities
)

# Compare the model with the historical reference
test_brier_skill = 1 - (
    test_brier / reference_brier
)

print("FY2019 Final Holdout — Probability Accuracy")
print(f"Training prevalence: {train_prevalence:.4%}")
print(f"Observed FY2019 incidence: {test_prevalence:.4%}")
print(f"Average predicted risk: {test_probabilities.mean():.4%}")
print(f"Reference Brier score: {reference_brier:.6f}")
print(f"Model Brier score: {test_brier:.6f}")
print(f"Brier Skill Score: {test_brier_skill:.4%}")

FY2019 Final Holdout — Probability Accuracy
Training prevalence: 3.7164%
Observed FY2019 incidence: 3.9107%
Average predicted risk: 6.1107%
Reference Brier score: 0.037581
Model Brier score: 0.037618
Brier Skill Score: -0.0984%


In [11]:
# Validate fixed review capacity
assert review_count == 4567

# Validate captured charge-offs
assert 0 <= captured_chargeoffs <= 1786

# Validate metric ranges
assert 0 <= test_ap <= 1
assert 0 <= test_roc_auc <= 1
assert 0 <= test_recall_10 <= 1
assert 0 <= test_precision_10 <= 1
assert 0 <= test_brier <= 1

# Check that numerical outputs are finite
assert np.isfinite([
    test_ap,
    test_roc_auc,
    test_recall_10,
    test_precision_10,
    test_brier,
    test_brier_skill
]).all()

print("Final FY2019 holdout metrics validated!")

Final FY2019 holdout metrics validated!


In [12]:
# Create a table of FY2019 outcomes and predicted risk
test_calibration_df = pd.DataFrame({
    "actual": y_test.to_numpy(),
    "predicted": test_probabilities
})

# Divide loans into ten approximately equal-sized risk groups
test_calibration_df["decile"] = (
    pd.qcut(
        test_calibration_df["predicted"].rank(method="first"),
        q=10,
        labels=False
    ) + 1
)

# Summarize predicted and observed risk
test_calibration_deciles = (
    test_calibration_df
    .groupby("decile")
    .agg(
        loan_count=("actual", "size"),
        chargeoffs=("actual", "sum"),
        avg_predicted_risk=("predicted", "mean"),
        observed_chargeoff_rate=("actual", "mean")
    )
    .reset_index()
)

# Calculate observed minus predicted probability
test_calibration_deciles["calibration_gap"] = (
    test_calibration_deciles["observed_chargeoff_rate"]
    - test_calibration_deciles["avg_predicted_risk"]
)

# Display percentages
display_df = test_calibration_deciles.copy()

percentage_columns = [
    "avg_predicted_risk",
    "observed_chargeoff_rate",
    "calibration_gap"
]

display_df[percentage_columns] *= 100

display(display_df.round(3))

# Validate the summary
assert len(test_calibration_deciles) == 10
assert test_calibration_deciles["loan_count"].sum() == 45670
assert test_calibration_deciles["chargeoffs"].sum() == 1786

print("\nFinal FY2019 calibration deciles validated!")

,decile,loan_count,chargeoffs,avg_predicted_risk,observed_chargeoff_rate,calibration_gap
0,1,4567,27,0.883,0.591,-0.292
1,2,4567,46,2.263,1.007,-1.256
2,3,4567,65,3.238,1.423,-1.815
3,4,4567,96,4.065,2.102,-1.963
4,5,4567,136,4.872,2.978,-1.895
5,6,4567,188,5.725,4.116,-1.609
6,7,4567,206,6.700,4.511,-2.190
7,8,4567,301,7.965,6.591,-1.375
8,9,4567,323,10.013,7.072,-2.941
9,10,4567,398,15.381,8.715,-6.667



Final FY2019 calibration deciles validated!


In [13]:
# Create output directory for evaluation results
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Store final FY2019 model performance
final_metrics = pd.DataFrame([{
    "Model": "Logistic Regression",
    "Evaluation_Period": "FY2019",
    "Test_Loans": len(y_test),
    "Chargeoffs": int(y_test.sum()),
    "Event_Prevalence": test_prevalence,
    "Average_Precision": test_ap,
    "AP_Lift": test_ap_lift,
    "ROC_AUC": test_roc_auc,
    "Recall_at_10": test_recall_10,
    "Precision_at_10": test_precision_10,
    "Precision_Lift": test_precision_lift,
    "Captured_Chargeoffs": captured_chargeoffs,
    "Review_Count": review_count,
    "Average_Predicted_Risk": test_probabilities.mean(),
    "Brier_Score": test_brier,
    "Reference_Brier": reference_brier,
    "Brier_Skill": test_brier_skill
}])

# Export final metrics
metrics_path = RESULTS_DIR / "fy2019_final_model_metrics.csv"

final_metrics.to_csv(
    metrics_path,
    index=False
)

print("Final metrics saved:", metrics_path.exists())

Final metrics saved: True


In [14]:
# Export the final calibration deciles
deciles_path = RESULTS_DIR / "fy2019_calibration_deciles.csv"

test_calibration_deciles.to_csv(
    deciles_path,
    index=False
)

print("Calibration deciles saved:", deciles_path.exists())

# Verify both files exist
assert metrics_path.exists()
assert deciles_path.exists()

print("\nFinal FY2019 evaluation results saved successfully!")

Calibration deciles saved: True

Final FY2019 evaluation results saved successfully!


In [17]:
# Verify modeling population
print("Total modeling records:", len(df))

print(
    "Unique loan IDs:",
    df["loan_row_id"].nunique()
)

print(
    "Duplicate loan IDs:",
    df["loan_row_id"].duplicated().sum()
)

print(
    "Five-year charge-off events:",
    int(df["chargeoff_within_5yr"].sum())
)

# Summarize loans by approval fiscal year
fy_summary = (
    df.groupby("approval_fy")
    .agg(
        loan_count=("loan_row_id", "size"),
        chargeoffs=("chargeoff_within_5yr", "sum")
    )
    .reset_index()
)

fy_summary["chargeoff_rate"] = (
    fy_summary["chargeoffs"] /
    fy_summary["loan_count"]
)

display(fy_summary.round(4))

# Validate our original modeling population
assert len(df) == 478953
assert df["loan_row_id"].nunique() == 478953
assert df["loan_row_id"].duplicated().sum() == 0
assert df["chargeoff_within_5yr"].sum() == 18826

print("\nStep 85.2 validation passed!")

Total modeling records: 478953
Unique loan IDs: 478953
Duplicate loan IDs: 0
Five-year charge-off events: 18826


,approval_fy,loan_count,chargeoffs,chargeoff_rate
0,2010,39912,1955,0.0490
1,2011,45628,1667,0.0365
2,2012,38889,1276,0.0328
3,2013,40416,1242,0.0307
4,2014,45961,1617,0.0352
5,2015,55420,2137,0.0386
6,2016,56785,2198,0.0387
7,2017,56078,2419,0.0431
8,2018,54194,2529,0.0467
9,2019,45670,1786,0.0391



Step 85.2 validation passed!
